# ChurnSignal — Milestone 5: Imbalance-Aware Model Evaluation
**Learning-First SaaS Customer Retention Intelligence**

Welcome to **Milestone 5**! Having trained our baseline models, we now master **imbalance-aware evaluation**.

---

### Core ML Principle:
> *"When a disease or churn affects only 1% of the population, standard metrics lie. We must measure performance using Precision, Recall, PR-AUC, and Operational Capacity (Precision@Top-K)."*

In this notebook, we explore:
1. **The Confusion Matrix**: True Positives, False Positives, True Negatives, False Negatives.
2. **ROC-AUC vs. PR-AUC**: Why ROC curves can be deceptively optimistic when negative samples dominate.
3. **Decision Threshold Tuning**: How shifting $\tau$ balances false alarms vs. missed churners.
4. **Precision@Top-K**: How our baseline delivers **9.5x higher churn detection** for a 50-call weekly retention team.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Add project root to sys.path
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import load_prediction_eligible_data
from src.features.engineer import engineer_all_features
from src.features.pipeline import chronological_split
from src.models.baselines import load_model
from src.evaluation.metrics import (
    compute_classification_metrics,
    compute_precision_at_k,
    compute_threshold_curve,
    find_optimal_threshold
)

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
print('Evaluation modules imported.')

## 1. Load Validation Data & Model Artifacts
We load our serialized feature pipeline and trained baseline model from `artifacts/`.


In [ ]:
raw_df = load_prediction_eligible_data()
df_feat = engineer_all_features(raw_df)
train_df, val_df, test_df = chronological_split(df_feat)

pipeline = load_model('feature_pipeline.joblib')
model = load_model('baseline_logistic_regression.joblib')

X_val = pipeline.transform(val_df)
y_val = val_df['churned_next_month'].values

# Compute continuous churn risk probabilities
y_prob = model.predict_proba(X_val)[:, 1]
print(f'Validation Records : {len(y_val):,}')
print(f'Actual Churners    : {y_val.sum():,} ({y_val.mean()*100:.2f}%)')

## 2. The Confusion Matrix

### Concept Intuition:
- **True Positives (TP)**: The model predicted churn, and the customer actually churned. (Success!)
- **False Positives (FP)**: The model predicted churn, but the customer stayed. (Wasted CS outreach time).
- **False Negatives (FN)**: The model predicted retention, but the customer churned! (Costliest business mistake: lost revenue).
- **True Negatives (TN)**: The model predicted retention, and the customer stayed.


In [ ]:
display(Image(filename='../reports/figures/07_confusion_matrix.png'))

opt_tau, best_f1 = find_optimal_threshold(y_val, y_prob, metric='f1')
y_pred_opt = (y_prob >= opt_tau).astype(int)

metrics = compute_classification_metrics(y_val, y_pred_opt, y_prob)
print(f'Optimal Threshold (max F1) : tau = {opt_tau:.2f}')
print(f'Accuracy                   : {metrics["accuracy"]*100:.2f}%')
print(f'Precision                  : {metrics["precision"]*100:.2f}%')
print(f'Recall                     : {metrics["recall"]*100:.2f}%')
print(f'F1 Score                   : {metrics["f1"]:.4f}')

## 3. ROC-AUC vs. PR-AUC: The Imbalance Reality

### Why ROC-AUC is Deceptive:
The False Positive Rate is defined as:
$$\text{FPR} = \frac{\text{FP}}{\text{FP} + \text{TN}}$$
Because there are **12,401 True Negatives**, even a large number of false alarms (e.g. 500 FPs) produces a tiny FPR of $\frac{500}{500 + 12401} \approx 0.038$ (3.8%). The ROC curve looks stunning (ROC-AUC = 0.807)!

### Why PR-AUC Tells the Truth:
Precision looks only at alerts:
$$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$
If you have 100 TPs and 500 FPs, your precision is only $\frac{100}{600} \approx 16.7\%$.
PR-AUC captures the real operational friction of finding needles in a haystack.


In [ ]:
display(Image(filename='../reports/figures/08_roc_and_pr_curves.png'))
print(f'ROC-AUC Score : {metrics["roc_auc"]:.4f}')
print(f'PR-AUC Score  : {metrics["pr_auc"]:.4f} (Base rate: {y_val.mean():.4f})')

## 4. Decision Threshold Tuning: Precision vs. Recall

### Concept Intuition:
As you lower $\tau$:
- **Recall goes UP**: You catch more churners.
- **Precision goes DOWN**: You trigger more false alarms.
- **Alert volume EXPLODES**: The team gets buried under alerts.


In [ ]:
display(Image(filename='../reports/figures/09_threshold_tradeoffs.png'))

thresh_table = compute_threshold_curve(y_val, y_prob, thresholds=[0.3, 0.5, 0.7, 0.83, 0.9])
display(thresh_table)

## 5. Precision@Top-K: Connecting ML to Team Bandwidth

### Business Operational Reality:
The Customer Success team can only review **50 accounts per week**.
Instead of applying an arbitrary threshold, we simply rank all accounts by predicted risk score and take the **Top 50**.

### Result:
- Out of the Top 50 highest-risk accounts, **16.0% actually churn**.
- Compared to picking customers at random (1.68%), our model provides **9.5x Lift**!


In [ ]:
display(Image(filename='../reports/figures/10_precision_at_k.png'))

k_results = [compute_precision_at_k(y_val, y_prob, k) for k in [25, 50, 100, 200, 500]]
display(pd.DataFrame(k_results))

## 6. Milestone 5 Summary & Transition to Model Comparison

### What We Mastered:
1. **The Metric Hierarchy**: Precision, Recall, and PR-AUC are the primary tools for rare-event churn prediction.
2. **Threshold Economics**: The best threshold depends on the cost of a false alarm vs. the cost of a lost customer.
3. **Precision@Top-K**: Direct alignment with CS team outreach bandwidth, yielding a **9.5x Lift over random outreach**.

### Next Step: Milestone 6 — Model Comparison
Can non-linear models (Decision Trees, Random Forests, LightGBM, and XGBoost) beat our Logistic Regression baseline? We find out next!
